Q9: Data Cleaning and Preparation

Analyzing customer churn in a telecommunications company.

Use Telecom_Customer_Churn.csv from the same folder. This is the complete original IBM Telco Customer Churn dataset with 7,043 rows and 21 columns.

Source: https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv

The original CSV is unchanged. Cleaned data is saved separately as Telecom_Customer_Churn_Cleaned.csv.

Imputation, outlier bounds and scaling are fitted on training data. For zero-tenure customers, MonthlyCharges is used as the average-charge fallback. The cleaned export is unscaled; model features are encoded and scaled separately.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("Telecom_Customer_Churn.csv")
print(data.head())
data.info()
print("Shape:", data.shape)

   customerID  gender  SeniorCitizen  ... MonthlyCharges TotalCharges  Churn
0  7590-VHVEG  Female              0  ...          29.85        29.85     No
1  5575-GNVDE    Male              0  ...          56.95       1889.5     No
2  3668-QPYBK    Male              0  ...          53.85       108.15    Yes
3  7795-CFOCW    Male              0  ...          42.30      1840.75     No
4  9237-HQITU  Female              0  ...          70.70       151.65    Yes

[5 rows x 21 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-

In [2]:
# Standardize text and convert data types
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].str.strip().replace("", np.nan)
data["gender"] = data["gender"].str.lower()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
print("Missing Values:")
print(data.isnull().sum())

data = data.drop_duplicates().copy()
print("After removing duplicates:", data.shape)

Missing Values:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64
After removing duplicates: (7043, 21)


In [3]:
# Split before fitting preprocessing values
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data["Churn"])
train = train.copy()
test = test.copy()
print("Training Shape:", train.shape)
print("Testing Shape:", test.shape)

Training Shape: (5634, 21)
Testing Shape: (1409, 21)


In [4]:
# Fill missing values using training data
num_cols = train.select_dtypes(include=np.number).columns
medians = train[num_cols].median()
train[num_cols] = train[num_cols].fillna(medians)
test[num_cols] = test[num_cols].fillna(medians)

cat_cols = train.select_dtypes(include="object").columns.drop(["customerID", "Churn"])
for col in cat_cols:
    mode = train[col].mode()[0]
    train[col] = train[col].fillna(mode)
    test[col] = test[col].fillna(mode)

for frame in [train, test]:
    frame["SeniorCitizen"] = frame["SeniorCitizen"].astype(int)
print("Training missing values:", train.isnull().sum().sum())
print("Testing missing values:", test.isnull().sum().sum())

Training missing values: 0
Testing missing values: 0


In [5]:
# Create new features and handle zero tenure separately
for frame in [train, test]:
    frame["AvgMonthlyCharge"] = frame["TotalCharges"] / frame["tenure"].replace(0, np.nan)
    frame["AvgMonthlyCharge"] = frame["AvgMonthlyCharge"].fillna(frame["MonthlyCharges"])
    frame["TenureGroup"] = pd.cut(
        frame["tenure"], bins=[0, 12, 24, 48, 100],
        labels=["New", "Medium", "Long", "Very Long"], include_lowest=True
    )

In [6]:
# Handle outliers using training IQR bounds
scale_cols = ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlyCharge"]
for col in scale_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(col, "training outliers:", ((train[col] < lower) | (train[col] > upper)).sum())
    train[col] = train[col].clip(lower, upper)
    test[col] = test[col].clip(lower, upper)

tenure training outliers: 0
MonthlyCharges training outliers: 0
TotalCharges training outliers: 0
AvgMonthlyCharge training outliers: 0


In [7]:
# Export cleaned records before encoding and scaling
cleaned = pd.concat([train, test]).sort_index()
cleaned.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)
print(cleaned.head())
print("Cleaned Shape:", cleaned.shape)
print("Saved Telecom_Customer_Churn_Cleaned.csv")

   customerID  gender  SeniorCitizen  ... Churn AvgMonthlyCharge  TenureGroup
0  7590-VHVEG  female              0  ...    No        29.850000          New
1  5575-GNVDE    male              0  ...    No        55.573529         Long
2  3668-QPYBK    male              0  ...   Yes        54.075000          New
3  7795-CFOCW    male              0  ...    No        40.905556         Long
4  9237-HQITU  female              0  ...   Yes        75.825000          New

[5 rows x 23 columns]
Cleaned Shape: (7043, 23)
Saved Telecom_Customer_Churn_Cleaned.csv


In [8]:
# Separate features and target, then encode categories
X_train = pd.get_dummies(train.drop(columns=["customerID", "Churn"]), drop_first=True, dtype=int)
X_test = pd.get_dummies(test.drop(columns=["customerID", "Churn"]), dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train["Churn"].map({"No": 0, "Yes": 1})
y_test = test["Churn"].map({"No": 0, "Yes": 1})

In [9]:
# Scale continuous features using training data
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])
print("Training Features:")
print(X_train.head())
print("\nTraining Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Features:
      SeniorCitizen    tenure  ...  TenureGroup_Long  TenureGroup_Very Long
3738              0  0.102371  ...                 1                      0
3151              0 -0.711743  ...                 0                      0
4860              0 -0.793155  ...                 0                      0
3867              0 -0.263980  ...                 1                      0
3810              0 -1.281624  ...                 0                      0

[5 rows x 34 columns]

Training Shape: (5634, 34)
Testing Shape: (1409, 34)
